In [5]:
# Lab hourly test: MI Cube TimePointValue (no interpolation)
# วนดึงทีละ 1 ชั่วโมง แล้วเก็บเฉพาะชั่วโมงที่มีค่าจริง
import os, math, json
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # reads .env from project root
API_TOKEN = os.environ["MI_CUBE_API_TOKEN"]
PI_SERVER = "VPKGDRBPPI01"
TP_URL = "https://micube.scg.com/v2/api/Datacenter/PIServer/TimePointValue/MultiTag/"
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
TAGS = [
    "PULP-LINE2-KAPPA-PRESS1",
    "PULP-LINE2-VISCOSITY-PRESS1",
    "PULP-LINE2-BRIGHTNESS-PRESS1",
]
DAY_START = datetime(2026, 10, 1, 0, 0, 0)
DAY_END = datetime(2026, 10, 1, 23, 0, 0)
OUT_DIR = Path(".")  # the notebook's own folder: Lab_Analysis/test pull data/

In [6]:
def fetch_timepoint(t):
    payload = {
        "token": API_TOKEN,
        "server": PI_SERVER,
        "tags": TAGS,
        "time_point": t.strftime("%Y-%m-%d %H:%M:%S"),
    }
    r = requests.post(TP_URL, headers=HEADERS, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()


def real_values(resp, t):
    rows = []

    for tag, obj in (resp.get("results") or {}).items():
        for it in obj if isinstance(obj, list) else [obj]:
            if not isinstance(it, dict):
                continue

            v = it.get("value")
            if v is None or isinstance(v, bool):
                continue

            try:
                v = float(v)
            except (TypeError, ValueError):
                continue

            if not math.isfinite(v):
                continue

            event_ts = it.get("timestamp")
            if not event_ts:
                continue

            # แปลงเวลาจาก API และเปรียบเทียบกับชั่วโมงที่ร้องขอ
            event_time = pd.to_datetime(event_ts)

            # ทำให้เป็น timezone-naive เพื่อเทียบกับ t ที่เป็น naive datetime
            if event_time.tzinfo is not None:
                event_time = event_time.tz_localize(None)

            # รับเฉพาะ Event ที่เกิดตรงกับเวลาที่ขอเท่านั้น
            if event_time != pd.Timestamp(t):
                continue

            rows.append({
                "hour": pd.Timestamp(t),
                "tag": tag,
                "timestamp": event_time,
                "value": v
            })

    return rows

In [7]:
raw, rows, errors = {}, [], []
for t in pd.date_range(DAY_START, DAY_END, freq="h"):
    try:
        raw[t] = fetch_timepoint(t)
    except Exception as e:
        errors.append((t, repr(e)))
        continue
    rows += real_values(raw[t], t)

# keep raw responses on disk so re-parsing never needs another API call
RAW_PATH = OUT_DIR / "timepoint_raw_20261001.json"
RAW_PATH.write_text(json.dumps({str(k): v for k, v in raw.items()}, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"requests: {len(raw) + len(errors)} | ok: {len(raw)} | errors: {len(errors)}")
for t, e in errors:
    print("ERROR", t, e)
if raw:
    first_t = next(iter(raw))
    print("Raw response sample at", first_t)
    print(json.dumps(raw[first_t], ensure_ascii=False, indent=2)[:1500])

requests: 24 | ok: 24 | errors: 0
Raw response sample at 2026-10-01 00:00:00
{
  "status": "success",
  "message": "Time point values retrieved successfully",
  "results": {
    "PULP-LINE2-KAPPA-PRESS1": {
      "timestamp": "2026-10-01T00:00:00",
      "value": "10.699999809265137",
      "description": "Kappa value from pulp line1",
      "unit": ""
    },
    "PULP-LINE2-VISCOSITY-PRESS1": {
      "timestamp": "2026-10-01T00:00:00",
      "value": "718.0",
      "description": "Viscosity value from pulp line1",
      "unit": ""
    },
    "PULP-LINE2-BRIGHTNESS-PRESS1": {
      "timestamp": "2026-10-01T00:00:00",
      "value": "36.099998474121094",
      "description": "Brightness value from pulp line1",
      "unit": ""
    }
  }
}


In [8]:
df = pd.DataFrame(rows)
print(f"hours with at least one real value: {df['hour'].nunique() if len(df) else 0} / 24")
if len(df):
    wide = df.pivot_table(index="hour", columns="tag", values="value", aggfunc="first").dropna(how="all")
    print(wide.to_string())
else:
    print("No real values returned for this day.")

hours with at least one real value: 24 / 24
tag                  PULP-LINE2-BRIGHTNESS-PRESS1  PULP-LINE2-KAPPA-PRESS1  PULP-LINE2-VISCOSITY-PRESS1
hour                                                                                                   
2026-10-01 00:00:00                     36.099998                   10.700                      718.000
2026-10-01 01:00:00                     36.650002                   10.600                      748.000
2026-10-01 02:00:00                     37.200001                   10.500                      778.000
2026-10-01 03:00:00                     36.599998                   10.400                      808.000
2026-10-01 04:00:00                     36.000000                   10.300                      838.000
2026-10-01 05:00:00                     36.049999                   10.425                      830.750
2026-10-01 06:00:00                     36.099998                   10.550                      823.500
2026-10-01 07:00:00 

In [9]:
# ===== Recorded values (ค่าจริงที่ lab บันทึก ไม่ interpolate) =====
# Step A: probe หา endpoint — read-only, 1 tag, ช่วงเวลาเดียวกับกราฟ PI Vision (14 Oct 15:38 -> 15 Oct 15:38)
# คาดหวัง Kappa: ~16:00 10.6, 20:00 12.1, 00:00 10.0, 08:00 11.0, 12:00 10.6 (ทุก 4 ชม., บางจุดอาจหาย)
BASE_URL = "https://micube.scg.com/v2/api/Datacenter/PIServer/"
CANDIDATES = ["RecordedValue", "RecordedValues", "CompressedValue", "CompressedValues", "RawValue", "RecordValue"]
REC_START = datetime(2026, 10, 14, 15, 38, 38)
REC_END = datetime(2026, 10, 15, 15, 38, 38)


def rec_payload(tags):
    return {
        "token": API_TOKEN,
        "server": PI_SERVER,
        "tags": tags,
        "start_time": REC_START.strftime("%Y-%m-%d %H:%M:%S"),
        "end_time": REC_END.strftime("%Y-%m-%d %H:%M:%S"),
    }


for name in CANDIDATES:
    url = f"{BASE_URL}{name}/MultiTag/"
    try:
        r = requests.post(url, headers=HEADERS, json=rec_payload(TAGS[:1]), timeout=30)
        print(f"{name:18s} {r.status_code}  {r.text[:300]}")
    except Exception as e:
        print(f"{name:18s} ERROR {e!r}")

RecordedValue      404  {
    "message": "The route api/Datacenter/PIServer/RecordedValue/MultiTag could not be found."
}
RecordedValues     404  {
    "message": "The route api/Datacenter/PIServer/RecordedValues/MultiTag could not be found."
}
CompressedValue    404  {
    "message": "The route api/Datacenter/PIServer/CompressedValue/MultiTag could not be found."
}
CompressedValues   404  {
    "message": "The route api/Datacenter/PIServer/CompressedValues/MultiTag could not be found."
}
RawValue           404  {
    "message": "The route api/Datacenter/PIServer/RawValue/MultiTag could not be found."
}
RecordValue        404  {
    "message": "The route api/Datacenter/PIServer/RecordValue/MultiTag could not be found."
}


In [10]:
# Step B: ดึงครั้งเดียวทั้งช่วง ด้วย endpoint ที่ Step A ได้ 200 + มีหลายจุดพร้อม timestamp จริง
REC_NAME = "RecordedValue"  # <- แก้เป็นชื่อที่ใช้ได้จาก Step A
REC_URL = f"{BASE_URL}{REC_NAME}/MultiTag/"

r = requests.post(REC_URL, headers=HEADERS, json=rec_payload(TAGS), timeout=120)
r.raise_for_status()
rec_raw = r.json()

REC_RAW_PATH = OUT_DIR / f"recorded_raw_{REC_START:%Y%m%d}.json"
REC_RAW_PATH.write_text(json.dumps(rec_raw, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(rec_raw, ensure_ascii=False, indent=2)[:1500])


def recorded_points(resp):
    # เก็บทุกจุดตาม timestamp ที่ API ส่งมา ไม่ปัด ไม่ fill
    rows = []
    for tag, obj in (resp.get("results") or {}).items():
        for it in obj if isinstance(obj, list) else [obj]:
            if not isinstance(it, dict) or isinstance(it.get("value"), bool):
                continue
            try:
                v = float(it.get("value"))
            except (TypeError, ValueError):
                continue
            if not math.isfinite(v) or not it.get("timestamp"):
                continue
            ts = pd.to_datetime(it["timestamp"])
            if ts.tzinfo is not None:
                ts = ts.tz_localize(None)
            rows.append({"tag": tag, "timestamp": ts, "value": v})
    return rows


rec = pd.DataFrame(recorded_points(rec_raw))
print(f"\nreal points: {len(rec)}")
if len(rec):
    print(rec.sort_values(["tag", "timestamp"]).to_string(index=False))

HTTPError: 404 Client Error: Not Found for url: https://micube.scg.com/v2/api/Datacenter/PIServer/RecordedValue/MultiTag/

In [ ]:
# Step C: มุมมองรายชั่วโมง — ชั่วโมงที่ไม่มีค่าจริงปล่อยว่าง (ไม่ interpolate)
if len(rec):
    rec["hour"] = rec["timestamp"].dt.floor("h")
    hours = pd.date_range(REC_START.replace(minute=0, second=0), REC_END, freq="h")
    hourly = rec.pivot_table(index="hour", columns="tag", values="value", aggfunc="first").reindex(hours)
    print(hourly.to_string())
    print("\nreal points per tag:")
    print(rec.groupby("tag").size().to_string())
else:
    print("No recorded values returned.")